# Applio Darwin — Notebook de production

Ce notebook est conçu pour un runtime Colab jetable avec GPU vérifié : le dépôt GitHub contient le code, Google Drive conserve les fichiers lourds et le runtime est reconstruit automatiquement à chaque session.

Le dépôt est déjà configuré pour `LizibaMvuluzi/applio-darwin`. Aucune URL GitHub n'est à modifier dans le notebook.

**Authentification GitHub :** le dépôt étant privé, la cellule de récupération utilise le secret Colab `GITHUB_TOKEN` s'il est disponible. Ce secret est stocké dans Colab, jamais dans le dépôt.


## 1. Récupérer automatiquement le projet depuis GitHub

In [ ]:
import os
import subprocess
from pathlib import Path

GITHUB_REPO = "https://github.com/LizibaMvuluzi/applio-darwin.git"
project_dir = Path("/content/applio-darwin")


def git_environment():
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN", "")

    if not token:
        raise RuntimeError(
            "Le dépôt GitHub est privé. Ajoute une seule fois le secret Colab "
            "GITHUB_TOKEN dans l’onglet 🔑 Secrets, active l’accès du notebook, "
            "puis relance cette cellule. Le token n’est jamais écrit dans Git."
        )

    askpass = Path("/tmp/applio_git_askpass.sh")
    askpass_script = r'''#!/bin/sh
case "$1" in
  *Username*) printf '%s\n' 'x-access-token' ;;
  *Password*) printf '%s\n' "$GITHUB_TOKEN" ;;
esac
'''
    askpass.write_text(askpass_script, encoding="utf-8")
    askpass.chmod(0o700)
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = str(askpass)
    return env


env = git_environment()

if not project_dir.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", GITHUB_REPO, str(project_dir)],
        env=env, check=True
    )
else:
    subprocess.run(
        ["git", "-C", str(project_dir), "remote", "set-url", "origin", GITHUB_REPO],
        env=env, check=True
    )
    subprocess.run(
        ["git", "-C", str(project_dir), "pull", "--ff-only"],
        env=env, check=True
    )

os.chdir(project_dir)
print("✅ Projet récupéré depuis GitHub.")


## 2. Connecter Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
print("✅ Drive monté.")


## 2 bis. Vérification GPU avant toute installation

Cette vérification ne lance aucun calcul d'inférence ni d'entraînement. Elle évite de télécharger tout l'environnement si Colab n'a pas réellement attribué de GPU.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ Aucun GPU CUDA attribué à ce runtime. Dans Colab : Runtime → Change runtime type → GPU, puis relance le notebook.")
print(f"✅ GPU Colab : {torch.cuda.get_device_name(0)}")
print(f"   PyTorch : {torch.__version__} · CUDA : {torch.version.cuda}")


## 3. Préparer automatiquement la configuration

In [ ]:
from pathlib import Path
import shutil

example = Path("config/config.example.json")
target = Path("config/config.json")

if not example.exists():
    raise FileNotFoundError(f"Modèle de configuration introuvable : {example}")

if not target.exists():
    shutil.copy2(example, target)
    print("✅ config/config.json créé automatiquement.")
else:
    print("✅ config/config.json déjà présent.")

print(target.read_text(encoding="utf-8"))


## 4. Installer automatiquement Applio

L'installation utilise un environnement Python 3.12 isolé. Le commit Applio est verrouillé automatiquement sur Google Drive après sa première résolution, afin que les sessions suivantes réutilisent exactement la même version.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "scripts/setup.py", "--config", "config/config.json"],
    text=True,
    capture_output=True,
)

print("===== SORTIE SETUP.PY =====")
print(result.stdout or "(vide)")
print("===== ERREUR SETUP.PY =====")
print(result.stderr or "(vide)")
print(f"===== CODE DE SORTIE : {result.returncode} =====")

if result.returncode != 0:
    raise RuntimeError(
        f"❌ setup.py a échoué avec le code {result.returncode}. "
        "La vraie erreur est affichée ci-dessus."
    )

print("✅ Installation Applio terminée.")


## 5. Diagnostic complet

Le diagnostic vérifie l'installation, les ressources, le GPU, le modèle Darwin, l'index et l'audio avant l'inférence.

In [ ]:
import json
from pathlib import Path

with open("config/config.json", encoding="utf-8") as f:
    cfg = json.load(f)

APPLIO_PYTHON = str(Path(cfg.get("runtime", {}).get("python_env_dir", "/content/applio-env")) / "bin" / "python")
print(f"Python Applio : {APPLIO_PYTHON}")

if not Path(APPLIO_PYTHON).is_file():
    raise FileNotFoundError(
        f"❌ Python Applio introuvable : {APPLIO_PYTHON}. "
        "L'installation doit être corrigée avant de poursuivre."
    )
print("✅ Interpréteur Applio présent.")


In [ ]:
import subprocess

result = subprocess.run(
    [APPLIO_PYTHON, "scripts/check_environment.py", "--config", "config/config.json"],
    text=True,
    capture_output=True,
)
print(result.stdout or "(vide)")
if result.stderr:
    print("--- STDERR ---")
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"❌ Diagnostic bloquant (code {result.returncode}).")


## 6. Inférence — Niveau 1

La conversion utilise automatiquement le modèle `darwin` et son index conservés sur Google Drive, puis sauvegarde le WAV et le log dans `ApplioExported/`.

In [ ]:
import subprocess

result = subprocess.run(
    [APPLIO_PYTHON, "scripts/inference.py", "--config", "config/config.json"],
    text=True,
    capture_output=True,
)
print(result.stdout or "(vide)")
if result.stderr:
    print("--- STDERR ---")
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"❌ Inférence échouée (code {result.returncode}).")


In [ ]:
# Écoute directement le résultat ici
import json
from pathlib import Path
from IPython.display import Audio, display

with open("config/config.json", encoding="utf-8") as f:
    cfg = json.load(f)

output_path = f"{cfg['drive']['root']}/{cfg['drive']['export_folder']}/{cfg['model_name']}_output.wav"
if not Path(output_path).exists():
    raise FileNotFoundError(f"Sortie introuvable : {output_path}")
display(Audio(output_path))


---
## Section Entraînement — Niveau 3

⚠️ **Ne pas exécuter avant d'avoir validé les Niveaux 1 et 2.**

Ces cellules sont préparées pour le futur entraînement du modèle `darwin`.

## 7. Entraînement Niveau 3 — EXPLICITEMENT DÉSACTIVÉ PAR DÉFAUT

✅ **« Tout exécuter » reste sans entraînement tant que `RUN_TRAINING = False`.** Le Niveau 3 ne se lance que si tu modifies volontairement cette variable et exécutes la cellule seule.

Le Niveau 3 exécute uniquement `preprocess → extract → train`. La CLI actuelle d’Applio génère ensuite l’index à la fin de `train`.


In [ ]:
from pathlib import Path
import subprocess

# SÉCURITÉ : laisser False. Le modifier uniquement pour lancer volontairement le Niveau 3.
RUN_TRAINING = False

if not RUN_TRAINING:
    print("🛑 Entraînement désactivé — aucun GPU ne sera consommé pour le Niveau 3.")
    print("   Pour l’activer volontairement : modifier RUN_TRAINING = True, puis exécuter cette cellule seule.")
else:
    for step in ("preprocess", "extract", "train"):
        print(f"\n===== NIVEAU 3 — {step.upper()} =====")
        result = subprocess.run(
            [APPLIO_PYTHON, "scripts/train.py", "--config", "config/config.json", "--step", step],
            text=True,
            capture_output=True,
        )
        print(result.stdout or "(vide)")
        if result.stderr:
            print("--- STDERR ---")
            print(result.stderr)
        if result.returncode != 0:
            raise RuntimeError(f"❌ Étape d’entraînement {step} échouée (code {result.returncode}).")
    print("\n✅ Entraînement terminé. L’index est généré par Applio à la fin de l’étape train.")
